# Chapter 4 — Tokenizer optimization
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch04_tokenizer_optimization.ipynb)

BPE optimization, caching, chunking, parallel preprocessing, and encode-path benchmarking. This chapter is CPU-oriented; T4 is not required.

In [ ]:
from collections import Counter
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor
import time, re
text=('the quick brown fox jumps over the lazy dog. '*10000).strip()
print(len(text))

## 1. Baseline pair counting

In [ ]:
def count_pairs(seq): return Counter(zip(seq,seq[1:]))
ids=list(text.encode())
t=time.perf_counter(); c=count_pairs(ids); print('baseline sec=',time.perf_counter()-t,'unique pairs=',len(c))

## 2. Pre-tokenization + cache
Repeated words are encoded once and reused. This is the main reason caching can help natural-language corpora.

In [ ]:
pieces=re.findall(r'\S+|\s+',text)
@lru_cache(maxsize=None)
def encode_piece(piece): return tuple(piece.encode())
t=time.perf_counter(); encoded=[encode_piece(p) for p in pieces]; dt=time.perf_counter()-t
print('pieces=',len(pieces),'cache=',encode_piece.cache_info(),'sec=',dt)

## 3. Chunking
Chunk boundaries let us process a large corpus incrementally instead of keeping every intermediate structure alive.

In [ ]:
def chunks(s,n=50000):
    for i in range(0,len(s),n): yield s[i:i+n]
t=time.perf_counter(); total=Counter()
for ch in chunks(text): total.update(count_pairs(list(ch.encode())))
print('chunked sec=',time.perf_counter()-t,'pairs=',len(total))

## 4. Parallel encode benchmark
Colab CPU topology varies, so parallel processing is a benchmark exercise rather than an assumption that more workers are always faster.

In [ ]:
parts=list(chunks(text,20000))
def work(s): return len(s.encode())
for workers in [1,2,4]:
    t=time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as ex: out=list(ex.map(work,parts))
    print(workers,'workers:',time.perf_counter()-t,'sec',sum(out))

## Checkpoint
Compare baseline, cache, chunking, and parallel execution. The goal is to identify which bottleneck is algorithmic, memory-related, or runtime-related.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch04